# 03 — Stage 1 Triage: XGBoost Filter

**Project:** HierFuse — Hierarchical PowerShell / LotL Detection with Learned Fusion  
**Stage:** 1 of 3 — fast lexical triage filter  
**Platform:** Kaggle Free Tier · **CPU only** (set accelerator = None)  
**Runtime:** ~25–35 minutes  
**Input:** `02-splits-dataset`  
**Output:** `/kaggle/working/triage/`

## What this notebook does

Trains the Stage 1 triage filter on lexical features extracted from raw PowerShell text.

**4 models trained × 3 seeds:**

| # | Model | Purpose |
|---|---|---|
| 1 | TF-IDF + Logistic Regression | Floor baseline |
| 2 | TF-IDF + Random Forest | Ensemble baseline |
| 3 | XGBoost (TF-IDF only) | Ablation |
| 4 | **XGBoost (TF-IDF + hand features)** | **Primary triage model** |

**Novel design:** The primary model's output probability + 20 hand-feature SHAP vector  
constitute the *triage-as-modality* input to Stage 3 fusion — HierFuse's first pillar.

**Outputs saved to `/kaggle/working/triage/`:**
- `triage_results.json` — all metrics across models × seeds × test variants
- `models/xgb_full_seed{42,1337,2024}.json` — XGBoost models
- `probs/{train,val,test,test_c2}_probs_seed{N}.npy` — triage-as-modality vectors
- `vectorizer/tfidf_vectorizer.pkl` — fitted TF-IDF (shared across seeds)


## Cell 0 — Paths, configuration, and dataset discovery

In [1]:
import os, json, logging, warnings
from pathlib import Path

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger('triage')

SEEDS = [42, 1337, 2024]

def find_splits_dir():
    input_root = Path('/kaggle/input')
    if input_root.exists():
        matches = list(input_root.rglob('train_seed42.parquet'))
        if matches: return matches[0].parent
    fallback = Path('/kaggle/working/splits')
    if fallback.exists() and (fallback/'train_seed42.parquet').exists():
        return fallback
    return None

SPLITS_DIR   = find_splits_dir()
if SPLITS_DIR is None:
    raise FileNotFoundError(
        'Split parquets not found. Attach 02-splits-dataset or run 02_splits.ipynb first.')

WEIGHTS_PATH = SPLITS_DIR / 'class_weights_per_seed.json'
if not WEIGHTS_PATH.exists():
    raise FileNotFoundError(f'class_weights_per_seed.json not found at {WEIGHTS_PATH}')

OUT_ROOT   = Path('/kaggle/working/triage')
MODELS_DIR = OUT_ROOT / 'models'
PROBS_DIR  = OUT_ROOT / 'probs'
VEC_DIR    = OUT_ROOT / 'vectorizer'
for d in [MODELS_DIR, PROBS_DIR, VEC_DIR]: d.mkdir(parents=True, exist_ok=True)

with open(WEIGHTS_PATH) as f: CLASS_WEIGHTS = json.load(f)

log.info(f'Splits dir : {SPLITS_DIR}')
log.info(f'Output root: {OUT_ROOT}')
log.info('Class weights:')
for seed in SEEDS:
    w = CLASS_WEIGHTS[str(seed)]
    n_neg = w.get('n_benign', w.get('n_negative','?'))
    n_pos = w.get('n_malicious', w.get('n_positive','?'))
    spw   = float(w.get('scale_pos_weight', w.get('scale_pos_weight_xgb', 1.0)))
    log.info(f'  Seed {seed}: n_benign={n_neg:,}, n_malicious={n_pos:,}, scale_pos_weight={spw:.4f}')
print('Cell 0 OK.')


03:52:44 | INFO | Splits dir : /kaggle/input/datasets/onkarkmane1501/02-splits-dataset/splits
03:52:44 | INFO | Output root: /kaggle/working/triage
03:52:44 | INFO | Class weights:
03:52:44 | INFO |   Seed 42: n_benign=6,089, n_malicious=6,087, scale_pos_weight=1.0003
03:52:44 | INFO |   Seed 1337: n_benign=6,089, n_malicious=6,087, scale_pos_weight=1.0003
03:52:44 | INFO |   Seed 2024: n_benign=6,089, n_malicious=6,087, scale_pos_weight=1.0003


Cell 0 OK.


## Cell 1 — Installs and imports

In [2]:
!pip install -q xgboost scikit-learn numpy pandas pyarrow

import gc, re, math, pickle, time
import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve,
)
import xgboost as xgb

GLOBAL_T0 = time.time()
print(f'XGBoost {xgb.__version__}  imports OK.')


03:52:50 | INFO | NumExpr defaulting to 4 threads.


XGBoost 3.2.0  imports OK.


## Cell 2 — Hand-crafted lexical features (20 signals)

These 20 features encode obfuscation signals from the PowerShell threat-intel literature  
(Bohannon Invoke-Obfuscation, PowerPeeler CCS 2024, Hendler/Dubin baselines).

**Double duty in the paper:**
1. Boost XGBoost triage F1 and TPR@1%FPR
2. Provide the interpretable auxiliary modality in Stage 3 fusion


In [3]:
def feat_length(s):            return float(len(s))
def feat_entropy(s):
    if not s: return 0.0
    freq = {c: s.count(c)/len(s) for c in set(s)}
    return float(-sum(p*math.log2(p) for p in freq.values() if p > 0))
def feat_upper_ratio(s):       return sum(c.isupper() for c in s) / max(len(s),1)
def feat_digit_ratio(s):       return sum(c.isdigit() for c in s) / max(len(s),1)
def feat_space_ratio(s):       return s.count(' ') / max(len(s),1)
def feat_special_ratio(s):     return sum(not c.isalnum() and c!=' ' for c in s) / max(len(s),1)
def feat_pipe_count(s):        return float(s.count('|'))
def feat_semicolon_count(s):   return float(s.count(';'))
def feat_backtick_count(s):    return float(s.count('`'))
def feat_paren_count(s):       return float(s.count('(')+s.count(')'))
def feat_has_base64(s):        return float(bool(re.search(r'[A-Za-z0-9+/]{20,}={0,2}', s)))
def feat_has_iex(s):           return float('iex' in s.lower() or 'invoke-expression' in s.lower())
def feat_has_webclient(s):     return float('webclient' in s.lower() or 'downloadstring' in s.lower())
def feat_has_enc(s):           return float(bool(re.search(r'-en[ce]\b', s, re.I)))
def feat_has_bypass(s):        return float('bypass' in s.lower())
def feat_has_hidden(s):        return float('hidden' in s.lower())
def feat_has_nop(s):           return float(bool(re.search(r'-nop\b|-noprofile\b', s, re.I)))
def feat_has_char_cast(s):     return float(bool(re.search(r'\[char\]', s, re.I)))
def feat_has_convert(s):       return float(bool(re.search(r'\[convert\]', s, re.I)))
def feat_unique_char_ratio(s): return len(set(s)) / max(len(s),1)

HAND_FEATS = [
    feat_length, feat_entropy, feat_upper_ratio, feat_digit_ratio,
    feat_space_ratio, feat_special_ratio, feat_pipe_count, feat_semicolon_count,
    feat_backtick_count, feat_paren_count, feat_has_base64, feat_has_iex,
    feat_has_webclient, feat_has_enc, feat_has_bypass, feat_has_hidden,
    feat_has_nop, feat_has_char_cast, feat_has_convert, feat_unique_char_ratio,
]
HAND_FEAT_NAMES = [f.__name__.replace('feat_','') for f in HAND_FEATS]

def extract_hand_features(texts):
    """Returns dense float32 array (n_samples, 20)."""
    return np.array([[f(t) for f in HAND_FEATS] for t in texts], dtype=np.float32)

print(f'Hand features ({len(HAND_FEATS)}): {HAND_FEAT_NAMES}')


Hand features (20): ['length', 'entropy', 'upper_ratio', 'digit_ratio', 'space_ratio', 'special_ratio', 'pipe_count', 'semicolon_count', 'backtick_count', 'paren_count', 'has_base64', 'has_iex', 'has_webclient', 'has_enc', 'has_bypass', 'has_hidden', 'has_nop', 'has_char_cast', 'has_convert', 'unique_char_ratio']


## Cell 3 — Evaluation helper

All metrics computed once, reused for every model × seed × test variant.  
F1 uses threshold=0.5. TPR@FPR=1% interpolated from the ROC curve.  
**Accuracy is deliberately omitted** from the primary results dict.


In [4]:
def evaluate(y_true, y_prob, label=''):
    """Return dict of metrics. y_prob is the probability of class 1 (malicious)."""
    y_true = np.asarray(y_true); y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= 0.5).astype(int)
    f1    = float(f1_score(y_true, y_pred, zero_division=0))
    auroc = float(roc_auc_score(y_true, y_prob))
    ap    = float(average_precision_score(y_true, y_prob))
    fpr_arr, tpr_arr, _ = roc_curve(y_true, y_prob)
    tpr_at_1fpr = float(np.interp(0.01, fpr_arr, tpr_arr))
    prec_arr, rec_arr, thresholds = precision_recall_curve(y_true, y_prob)
    f1_arr      = 2*prec_arr*rec_arr / np.maximum(prec_arr+rec_arr, 1e-9)
    best_thresh = float(thresholds[np.argmax(f1_arr[:-1])]) if len(thresholds) else 0.5
    f1_opt      = float(f1_score(y_true,(y_prob>=best_thresh).astype(int),zero_division=0))
    result = {
        'f1':          round(f1,4), 'f1_opt': round(f1_opt,4),
        'best_thresh': round(best_thresh,4), 'auroc': round(auroc,4),
        'pr_auc':      round(ap,4), 'tpr_at_1fpr': round(tpr_at_1fpr,4),
        'n_samples':   int(len(y_true)), 'n_malicious': int(y_true.sum()),
        'mal_prev':    round(float(y_true.mean()),4),
    }
    if label:
        log.info(f'  [{label}] F1={result["f1"]:.4f}  F1-opt={result["f1_opt"]:.4f}  '
                 f'AUROC={result["auroc"]:.4f}  PR-AUC={result["pr_auc"]:.4f}  '
                 f'TPR@1%FPR={result["tpr_at_1fpr"]:.4f}')
    return result

print('Evaluation helper ready.')


Evaluation helper ready.


## Cell 4 — Fit TF-IDF vectorizer (seed-42 train, then applied to all seeds)

**Design choices:**
- `max_features=3_000` — down from 50k; Kaggle CPU with 35,686 × 50k causes OOM; 3k sufficient
- `analyzer='char_wb'`, `ngram_range=(2,4)` — matches Hendler et al. / Mimura et al. baselines
- `sublinear_tf=True` — log-scaling dampens outlier term frequencies
- Fit once, saved to disk, reused by NB07 final evaluation


In [5]:
log.info('Loading seed-42 train for TF-IDF fit...')
t0 = time.time()
train42 = pd.read_parquet(SPLITS_DIR/'train_seed42.parquet', columns=['script_text','label'])
log.info(f'  {len(train42):,} records loaded in {time.time()-t0:.1f}s')

TFIDF = TfidfVectorizer(
    analyzer     = 'char_wb',
    ngram_range  = (2, 4),
    max_features = 3_000,    # RAM-safe on 16 GB Kaggle CPU
    sublinear_tf = True,
    min_df       = 3,
    strip_accents= 'unicode',
)
TFIDF.fit(train42['script_text'])
log.info(f'Vocabulary size: {len(TFIDF.vocabulary_):,}  ({time.time()-t0:.1f}s)')

vec_path = VEC_DIR / 'tfidf_vectorizer.pkl'
with open(vec_path,'wb') as f: pickle.dump(TFIDF, f, protocol=4)
log.info(f'Vectorizer saved: {vec_path}')
del train42; gc.collect()
print('Cell 4 OK — TF-IDF fitted and saved.')


03:52:54 | INFO | Loading seed-42 train for TF-IDF fit...
03:53:00 | INFO |   12,176 records loaded in 6.7s
04:10:39 | INFO | Vocabulary size: 3,000  (1065.0s)
04:10:39 | INFO | Vectorizer saved: /kaggle/working/triage/vectorizer/tfidf_vectorizer.pkl


Cell 4 OK — TF-IDF fitted and saved.


## Cell 5 — Train all models across 3 seeds

For each seed:
1. Load train/val/test/test_c2 parquets
2. Extract TF-IDF sparse matrix + hand features dense array
3. Train LR → RF → XGBoost (TF-IDF only) → XGBoost (TF-IDF + hand)
4. Evaluate on balanced test **and** C2-compliant test
5. Save XGBoost model + triage-as-modality probability arrays

**XGBoost config:** `scale_pos_weight ≈ 1.0` because corpus is 50:50.  
`tree_method='hist'` is CPU-safe (not `device='cuda'`).


In [6]:
all_results = {}

for seed in SEEDS:
    log.info(f'\n{"="*60}\nSEED {seed}\n{"="*60}')
    t_seed = time.time()

    w   = CLASS_WEIGHTS[str(seed)]
    spw = float(w.get('scale_pos_weight', w.get('scale_pos_weight_xgb', 1.0)))
    log.info(f'  scale_pos_weight={spw:.4f}')

    cols = ['script_text','label']
    df_tr   = pd.read_parquet(SPLITS_DIR/f'train_seed{seed}.parquet',   columns=cols)
    df_va   = pd.read_parquet(SPLITS_DIR/f'val_seed{seed}.parquet',     columns=cols)
    df_te   = pd.read_parquet(SPLITS_DIR/f'test_seed{seed}.parquet',    columns=cols)
    df_tec2 = pd.read_parquet(SPLITS_DIR/f'test_c2_seed{seed}.parquet', columns=cols)

    y_tr=df_tr['label'].values; y_va=df_va['label'].values
    y_te=df_te['label'].values; y_tec2=df_tec2['label'].values

    log.info(f'  train={len(df_tr):,}  val={len(df_va):,}  test={len(df_te):,}  test_c2={len(df_tec2):,}')

    # TF-IDF
    X_tr_tfidf   = TFIDF.transform(df_tr['script_text'])
    X_va_tfidf   = TFIDF.transform(df_va['script_text'])
    X_te_tfidf   = TFIDF.transform(df_te['script_text'])
    X_tec2_tfidf = TFIDF.transform(df_tec2['script_text'])

    # Hand features
    H_tr   = extract_hand_features(df_tr['script_text'].tolist())
    H_va   = extract_hand_features(df_va['script_text'].tolist())
    H_te   = extract_hand_features(df_te['script_text'].tolist())
    H_tec2 = extract_hand_features(df_tec2['script_text'].tolist())

    # Combined (TF-IDF dense + hand) — 3020 features, safe in memory
    X_tr_full   = np.hstack([X_tr_tfidf.toarray(),   H_tr])
    X_va_full   = np.hstack([X_va_tfidf.toarray(),   H_va])
    X_te_full   = np.hstack([X_te_tfidf.toarray(),   H_te])
    X_tec2_full = np.hstack([X_tec2_tfidf.toarray(), H_tec2])
    for df in [df_tr, df_va, df_te, df_tec2]: del df['script_text']
    gc.collect()

    seed_results = {}

    # ── Model 1/4: Logistic Regression ───────────────────────────────────
    log.info('  [1/4] Logistic Regression...')
    lr = LogisticRegression(max_iter=1000, class_weight='balanced',
                             C=1.0, solver='saga', n_jobs=-1, random_state=seed)
    lr.fit(X_tr_tfidf, y_tr)
    seed_results['lr'] = {
        'balanced_test': evaluate(y_te,   lr.predict_proba(X_te_tfidf)[:,1],   'LR balanced'),
        'c2_test':       evaluate(y_tec2, lr.predict_proba(X_tec2_tfidf)[:,1], 'LR C2'),
    }
    del lr; gc.collect()

    # ── Model 2/4: Random Forest ──────────────────────────────────────────
    log.info('  [2/4] Random Forest...')
    rf = RandomForestClassifier(n_estimators=300, max_depth=None, class_weight='balanced',
                                 n_jobs=-1, random_state=seed)
    rf.fit(X_tr_tfidf, y_tr)
    seed_results['rf'] = {
        'balanced_test': evaluate(y_te,   rf.predict_proba(X_te_tfidf)[:,1],   'RF balanced'),
        'c2_test':       evaluate(y_tec2, rf.predict_proba(X_tec2_tfidf)[:,1], 'RF C2'),
    }
    del rf; gc.collect()

    # ── Model 3/4: XGBoost (TF-IDF only) ─────────────────────────────────
    log.info('  [3/4] XGBoost (TF-IDF only)...')
    xgb_tfidf = xgb.XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw,
        eval_metric='logloss', use_label_encoder=False,
        random_state=seed, n_jobs=-1, tree_method='hist')
    xgb_tfidf.fit(X_tr_tfidf, y_tr, eval_set=[(X_va_tfidf, y_va)], verbose=False)
    seed_results['xgb_tfidf'] = {
        'balanced_test': evaluate(y_te,   xgb_tfidf.predict_proba(X_te_tfidf)[:,1],   'XGB-tfidf balanced'),
        'c2_test':       evaluate(y_tec2, xgb_tfidf.predict_proba(X_tec2_tfidf)[:,1], 'XGB-tfidf C2'),
    }
    del xgb_tfidf; gc.collect()

    # ── Model 4/4: XGBoost (TF-IDF + hand) — PRIMARY ─────────────────────
    log.info('  [4/4] XGBoost (TF-IDF + hand features) — PRIMARY...')
    xgb_full = xgb.XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw,
        eval_metric='logloss', use_label_encoder=False,
        random_state=seed, n_jobs=-1, tree_method='hist')
    xgb_full.fit(X_tr_full, y_tr, eval_set=[(X_va_full, y_va)], verbose=False)

    xf_te   = xgb_full.predict_proba(X_te_full)[:,1]
    xf_tec2 = xgb_full.predict_proba(X_tec2_full)[:,1]
    xf_tr   = xgb_full.predict_proba(X_tr_full)[:,1]   # triage-as-modality
    xf_va   = xgb_full.predict_proba(X_va_full)[:,1]

    fi_all  = xgb_full.feature_importances_
    fi_hand = fi_all[-len(HAND_FEATS):]
    seed_results['xgb_full'] = {
        'balanced_test': evaluate(y_te,   xf_te,   'XGB-full balanced'),
        'c2_test':       evaluate(y_tec2, xf_tec2, 'XGB-full C2'),
        'hand_feature_importances': dict(zip(HAND_FEAT_NAMES, fi_hand.tolist())),
    }

    xgb_full.save_model(str(MODELS_DIR/f'xgb_full_seed{seed}.json'))
    np.save(PROBS_DIR/f'train_probs_seed{seed}.npy',   xf_tr)
    np.save(PROBS_DIR/f'val_probs_seed{seed}.npy',     xf_va)
    np.save(PROBS_DIR/f'test_probs_seed{seed}.npy',    xf_te)
    np.save(PROBS_DIR/f'test_c2_probs_seed{seed}.npy', xf_tec2)
    log.info(f'  Seed {seed} complete — {(time.time()-t_seed)/60:.1f} min')

    all_results[str(seed)] = seed_results
    del xgb_full, X_tr_full, X_va_full, X_te_full, X_tec2_full
    del X_tr_tfidf, X_va_tfidf, X_te_tfidf, X_tec2_tfidf
    del H_tr, H_va, H_te, H_tec2
    gc.collect()

results_path = OUT_ROOT / 'triage_results.json'
with open(results_path,'w') as f: json.dump(all_results, f, indent=2)
log.info(f'Results saved: {results_path}')
log.info(f'Total wall time: {(time.time()-GLOBAL_T0)/60:.1f} min')
print('Cell 5 OK.')


04:10:39 | INFO | 
SEED 42
04:10:39 | INFO |   scale_pos_weight=1.0003
04:10:45 | INFO |   train=12,176  val=2,610  test=2,610  test_c2=1,450
04:35:05 | INFO |   [1/4] Logistic Regression...
04:35:07 | INFO |   [LR balanced] F1=0.9257  F1-opt=0.9353  AUROC=0.9817  PR-AUC=0.9826  TPR@1%FPR=0.7333
04:35:07 | INFO |   [LR C2] F1=0.7939  F1-opt=0.8251  AUROC=0.9824  PR-AUC=0.9033  TPR@1%FPR=0.7241
04:35:07 | INFO |   [2/4] Random Forest...
04:35:58 | INFO |   [RF balanced] F1=0.9342  F1-opt=0.9493  AUROC=0.9877  PR-AUC=0.9885  TPR@1%FPR=0.8431
04:35:59 | INFO |   [RF C2] F1=0.8291  F1-opt=0.8643  AUROC=0.9885  PR-AUC=0.9284  TPR@1%FPR=0.8072
04:35:59 | INFO |   [3/4] XGBoost (TF-IDF only)...
04:40:28 | INFO |   [XGB-tfidf balanced] F1=0.9543  F1-opt=0.9592  AUROC=0.9920  PR-AUC=0.9924  TPR@1%FPR=0.8705
04:40:28 | INFO |   [XGB-tfidf C2] F1=0.8344  F1-opt=0.8944  AUROC=0.9917  PR-AUC=0.9514  TPR@1%FPR=0.8759
04:40:28 | INFO |   [4/4] XGBoost (TF-IDF + hand features) — PRIMARY...
04:44:49 | 

Cell 5 OK.


## Cell 6 — Results table (Table 1, Stage 1 triage rows)

In [7]:
MODELS      = ['lr','rf','xgb_tfidf','xgb_full']
MODEL_NAMES = ['LR (TF-IDF)','RF (TF-IDF)','XGBoost (TF-IDF)','XGBoost (TF-IDF+hand)']
METRICS     = ['f1','auroc','pr_auc','tpr_at_1fpr']
METRIC_NAMES= ['F1','AUROC','PR-AUC','TPR@1%FPR']
VARIANTS    = [('balanced_test','Balanced test (~50% mal)'),
               ('c2_test',      'C2 test (≤10% mal)')]

for variant_key, variant_label in VARIANTS:
    print(f'\n=== {variant_label} ===')
    header = f'{"Model":<28}' + ''.join(f'{m:>18}' for m in METRIC_NAMES)
    print(header); print('-'*len(header))
    for model_key, model_name in zip(MODELS, MODEL_NAMES):
        row = []
        for metric in METRICS:
            vals = [all_results[str(s)][model_key][variant_key][metric] for s in SEEDS]
            row.append(f'{np.mean(vals):.4f}±{np.std(vals):.4f}')
        print(f'{model_name:<28}' + ''.join(f'{v:>18}' for v in row))

print()
print('Note: XGBoost (TF-IDF+hand) is the primary triage model.')
print('Triage probabilities in /kaggle/working/triage/probs/ are the')
print('triage-as-modality inputs to Stage 3 fusion.')



=== Balanced test (~50% mal) ===
Model                                       F1             AUROC            PR-AUC         TPR@1%FPR
----------------------------------------------------------------------------------------------------
LR (TF-IDF)                      0.9301±0.0049     0.9821±0.0004     0.9829±0.0004     0.7443±0.0177
RF (TF-IDF)                      0.9408±0.0047     0.9873±0.0004     0.9881±0.0004     0.8298±0.0099
XGBoost (TF-IDF)                 0.9581±0.0030     0.9919±0.0003     0.9923±0.0003     0.8736±0.0025
XGBoost (TF-IDF+hand)            0.9591±0.0030     0.9927±0.0003     0.9930±0.0002     0.8835±0.0080

=== C2 test (≤10% mal) ===
Model                                       F1             AUROC            PR-AUC         TPR@1%FPR
----------------------------------------------------------------------------------------------------
LR (TF-IDF)                      0.7983±0.0069     0.9841±0.0030     0.9108±0.0054     0.7425±0.0172
RF (TF-IDF)                  

## Cell 7 — Feature importances

In [8]:
fi_agg = {name:[] for name in HAND_FEAT_NAMES}
for seed in SEEDS:
    fi = all_results[str(seed)]['xgb_full'].get('hand_feature_importances',{})
    for name in HAND_FEAT_NAMES: fi_agg[name].append(fi.get(name,0.0))

fi_mean = {name: float(np.mean(vals)) for name,vals in fi_agg.items()}
fi_sorted = sorted(fi_mean.items(), key=lambda x: x[1], reverse=True)

print('Top 20 hand-feature importances (mean across 3 seeds):')
print(f'{"Feature":<28}  Importance')
print('-'*42)
for feat_name, imp in fi_sorted:
    bar = '█' * int(imp*400)
    print(f'{feat_name:<28}  {imp:.6f}  {bar}')


Top 20 hand-feature importances (mean across 3 seeds):
Feature                       Importance
------------------------------------------
digit_ratio                   0.044365  █████████████████
has_webclient                 0.014752  █████
has_convert                   0.009493  ███
unique_char_ratio             0.002470  
has_iex                       0.002221  
length                        0.001519  
has_hidden                    0.001370  
semicolon_count               0.000986  
has_bypass                    0.000975  
space_ratio                   0.000621  
has_nop                       0.000568  
upper_ratio                   0.000458  
has_char_cast                 0.000437  
has_base64                    0.000365  
entropy                       0.000317  
pipe_count                    0.000294  
paren_count                   0.000280  
special_ratio                 0.000257  
backtick_count                0.000188  
has_enc                       0.000000  


## Cell 8 — Sanity checks

In [9]:
checks_passed = checks_failed = 0
def check(condition, label, detail=''):
    global checks_passed, checks_failed
    if condition: checks_passed += 1; print(f'[OK]   {label}')
    else:         checks_failed += 1; print(f'[FAIL] {label}'); print(f'       {detail}') if detail else None

print('\n=== SANITY CHECKS ===')

check((OUT_ROOT/'triage_results.json').exists(), 'triage_results.json saved')
check((VEC_DIR/'tfidf_vectorizer.pkl').exists(), 'TF-IDF vectorizer saved')

for seed in SEEDS:
    check((MODELS_DIR/f'xgb_full_seed{seed}.json').exists(), f'Seed {seed}: XGBoost model saved')
    for split in ['train','val','test','test_c2']:
        p = PROBS_DIR/f'{split}_probs_seed{seed}.npy'
        check(p.exists(), f'Seed {seed}: {split} probs saved')
        if p.exists():
            arr = np.load(p)
            check(arr.min()>=0 and arr.max()<=1,
                  f'Seed {seed}: {split} probs in [0,1]',
                  f'min={arr.min():.4f}, max={arr.max():.4f}')
    for vkey in ['balanced_test','c2_test']:
        m = all_results[str(seed)]['xgb_full'][vkey]
        check(m['auroc']>0.90, f'Seed {seed} xgb_full {vkey}: AUROC>0.90', f'AUROC={m["auroc"]:.4f}')
        check(m['f1']>0.70,   f'Seed {seed} xgb_full {vkey}: F1>0.70',    f'F1={m["f1"]:.4f}')

print(f'\nResults: {checks_passed} passed, {checks_failed} failed')
if checks_failed == 0:
    print('\nALL CHECKS PASSED.')
    print('\nNext: upload /kaggle/working/triage/ as Kaggle Dataset "03-triage-dataset"')
    print('Then run 04_nlp_branch.ipynb and 05_gat_branch.ipynb (can run in parallel)')
else:
    raise RuntimeError(f'{checks_failed} checks FAILED — review above before proceeding.')



=== SANITY CHECKS ===
[OK]   triage_results.json saved
[OK]   TF-IDF vectorizer saved
[OK]   Seed 42: XGBoost model saved
[OK]   Seed 42: train probs saved
[OK]   Seed 42: train probs in [0,1]
[OK]   Seed 42: val probs saved
[OK]   Seed 42: val probs in [0,1]
[OK]   Seed 42: test probs saved
[OK]   Seed 42: test probs in [0,1]
[OK]   Seed 42: test_c2 probs saved
[OK]   Seed 42: test_c2 probs in [0,1]
[OK]   Seed 42 xgb_full balanced_test: AUROC>0.90
[OK]   Seed 42 xgb_full balanced_test: F1>0.70
[OK]   Seed 42 xgb_full c2_test: AUROC>0.90
[OK]   Seed 42 xgb_full c2_test: F1>0.70
[OK]   Seed 1337: XGBoost model saved
[OK]   Seed 1337: train probs saved
[OK]   Seed 1337: train probs in [0,1]
[OK]   Seed 1337: val probs saved
[OK]   Seed 1337: val probs in [0,1]
[OK]   Seed 1337: test probs saved
[OK]   Seed 1337: test probs in [0,1]
[OK]   Seed 1337: test_c2 probs saved
[OK]   Seed 1337: test_c2 probs in [0,1]
[OK]   Seed 1337 xgb_full balanced_test: AUROC>0.90
[OK]   Seed 1337 xgb_full